In [ ]:
# Install required packages
%pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn opencv-python pillow

# EfficientNetB0 Cataract Detection Training

This notebook trains an EfficientNetB0 model for cataract detection.
The trained model will be used as a second opinion alongside the ResNet50 model.

In [ ]:
# Import libraries
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import os
from PIL import Image
from tqdm import tqdm

# IMPORTANT: DISABLE mixed precision for EfficientNetB0
# EfficientNet has batch normalization layers that cause numerical instability with float16
tf.keras.mixed_precision.set_global_policy('float32')

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("Precision: float32 (stable for EfficientNetB0)")
print("⚠️  Mixed precision DISABLED - EfficientNetB0 requires float32 for stability")

## 1. Configuration

In [ ]:
# Configuration
RAW_DATASET_PATH = Path('/kaggle/input/datasets/nicolenankabruce/dataset-12/images')  # UPDATE THIS - your raw data
PREPROCESSED_PATH = Path('/kaggle/working/preprocessed_efficientnet')  # Where preprocessed images will be saved

IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Reduced from 64 for stability
EPOCHS_PHASE1 = 15  # Increased from 10
EPOCHS_PHASE2 = 20  # Increased from 15
LEARNING_RATE_PHASE1 = 0.01  # Increased from 0.001 - EfficientNet needs higher LR
LEARNING_RATE_PHASE2 = 0.001  # Increased from 0.0001

# Model save paths
KERAS_MODEL_PATH = 'efficientnetb0_cataract.keras'
TFLITE_PATH = 'efficientnetb0_cataract.tflite'
TFLITE_FLOAT16_PATH = 'efficientnetb0_cataract_float16.tflite'

print(f"Raw dataset path: {RAW_DATASET_PATH}")
print(f"Preprocessed path: {PREPROCESSED_PATH}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rates: Phase1={LEARNING_RATE_PHASE1}, Phase2={LEARNING_RATE_PHASE2}")

## 2. Enhanced Preprocessing Pipeline

In [ ]:
class EyePreprocessor:
    """
    Comprehensive preprocessing pipeline for eye images.
    NOTE: This will be run ONCE before training, not during training.
    """

    def __init__(self, target_size=(224, 224)):
        self.target_size = target_size

    def detect_and_mask_reflections(self, image):
        """Detect and mask specular highlights (flash artifacts)."""
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        lower_highlight = np.array([0, 0, 200])
        upper_highlight = np.array([180, 50, 255])
        highlight_mask = cv2.inRange(hsv, lower_highlight, upper_highlight)
        kernel = np.ones((3, 3), np.uint8)
        highlight_mask = cv2.dilate(highlight_mask, kernel, iterations=1)
        result = cv2.inpaint(image, highlight_mask, 3, cv2.INPAINT_TELEA)
        return result

    def detect_iris_region(self, image):
        """Detect and crop the iris/pupil region."""
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (9, 9), 2)
        circles = cv2.HoughCircles(
            blurred,
            cv2.HOUGH_GRADIENT,
            dp=1,
            minDist=100,
            param1=50,
            param2=30,
            minRadius=30,
            maxRadius=150
        )
        
        if circles is not None:
            circles = np.uint16(np.around(circles))
            x, y, r = circles[0][0]
            
            margin = int(r * 1.5)
            x1 = max(0, int(x) - margin)
            y1 = max(0, int(y) - margin)
            x2 = min(image.shape[1], int(x) + margin)
            y2 = min(image.shape[0], int(y) + margin)
            
            if x2 > x1 and y2 > y1:
                cropped = image[y1:y2, x1:x2]
                if cropped.size > 0:
                    return cropped
        
        return image

    def apply_clahe_rgb(self, image):
        """Apply CLAHE with safety check."""
        if image is None or image.size == 0:
            raise ValueError("Empty image passed to CLAHE")
        
        image = image.astype(np.uint8)
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        return enhanced

    def is_blurry(self, image, threshold=100):
        """Check if image is blurry using Laplacian variance."""
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        return laplacian_var < threshold, laplacian_var

    def preprocess_image(self, image_path, apply_quality_checks=True):
        """
        Complete preprocessing pipeline.
        """
        img = cv2.imread(str(image_path))
        if img is None:
            return None, {'error': 'Failed to load image'}

        # Quality checks
        metadata = {}
        if apply_quality_checks:
            is_blurry, blur_score = self.is_blurry(img)
            metadata['blur_score'] = blur_score
            if is_blurry:
                metadata['warning'] = 'Image may be blurry'

        # Step 1: Reflection removal
        img = self.detect_and_mask_reflections(img)

        # Step 2: Iris region detection
        img = self.detect_iris_region(img)

        # Step 3: CLAHE enhancement
        img = self.apply_clahe_rgb(img)

        # Step 4: Resize
        img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_LANCZOS4)

        return img, metadata

## 3. ONE-TIME Preprocessing (Run Once, Save Results)

In [ ]:
def preprocess_dataset(raw_path, output_path):
    """
    Preprocess entire dataset ONCE and save to disk.
    This runs BEFORE training, not during.
    """
    preprocessor = EyePreprocessor(target_size=IMG_SIZE)
    
    # Process both train and test sets
    for split in ['Train', 'Test']:
        input_dir = raw_path / split
        output_dir = output_path / split.lower()  # EfficientNet expects lowercase
        
        if not input_dir.exists():
            print(f"⚠️ Warning: {input_dir} does not exist. Skipping {split} set.")
            continue
            
        print(f"\n{'='*80}")
        print(f"Processing {split.upper()} set...")
        print(f"{'='*80}")
        
        # Get all class directories
        class_dirs = [d for d in input_dir.iterdir() if d.is_dir()]
        
        total_processed = 0
        total_skipped = 0
        
        for class_dir in class_dirs:
            class_name = class_dir.name
            output_class_dir = output_dir / class_name
            output_class_dir.mkdir(parents=True, exist_ok=True)
            
            # Get all image files
            image_files = list(class_dir.glob('*.jpg')) + \
                         list(class_dir.glob('*.png')) + \
                         list(class_dir.glob('*.jpeg'))
            
            print(f"\nProcessing class '{class_name}': {len(image_files)} images")
            
            for img_path in tqdm(image_files, desc=f"  {class_name}"):
                output_path_img = output_class_dir / f"{img_path.stem}.jpg"
                
                # Skip if already processed
                if output_path_img.exists():
                    total_skipped += 1
                    continue
                
                # Preprocess
                processed_img, metadata = preprocessor.preprocess_image(img_path)
                
                if processed_img is not None:
                    cv2.imwrite(str(output_path_img), processed_img)
                    total_processed += 1
                else:
                    print(f"  ⚠️ Failed to process: {img_path.name}")
                    total_skipped += 1
        
        print(f"\n✅ {split.upper()} set complete:")
        print(f"   - Processed: {total_processed} images")
        print(f"   - Skipped: {total_skipped} images")

# Run preprocessing
print("🚀 Starting ONE-TIME preprocessing...")
print("This may take several minutes, but you only do it ONCE!")
print(f"\nInput: {RAW_DATASET_PATH}")
print(f"Output: {PREPROCESSED_PATH}")

start_time = time.time()
preprocess_dataset(RAW_DATASET_PATH, PREPROCESSED_PATH)
preprocess_time = (time.time() - start_time) / 60

print(f"\n{'='*80}")
print(f"✅ PREPROCESSING COMPLETE in {preprocess_time:.2f} minutes")
print(f"{'='*80}")

## 4. Create Optimized Data Generators

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    shear_range=0.1,
    fill_mode='nearest'
)

# Test data (only rescaling, no augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
print("Creating data generators from preprocessed images...")

train_generator = train_datagen.flow_from_directory(
    PREPROCESSED_PATH / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    PREPROCESSED_PATH / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\n✅ Data generators created:")
print(f"   Training samples: {train_generator.samples}")
print(f"   Test samples: {test_generator.samples}")
print(f"   Number of classes: {len(train_generator.class_indices)}")
print(f"   Class indices: {train_generator.class_indices}")
print(f"   Batch size: {BATCH_SIZE}")

## 5. Build EfficientNetB0 Model

In [ ]:
def create_model(num_classes):
    """
    Create EfficientNetB0 model with custom top layers.
    """
    # Load pre-trained EfficientNetB0 (without top layers)
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Add custom top layers with BatchNormalization for stability
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)  # BN for stability
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)  # BN for stability
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    
    return model, base_model

# Create model
num_classes = len(train_generator.class_indices)
model, base_model = create_model(num_classes)

print("✅ EfficientNetB0 model created")
print(f"   Total parameters: {model.count_params():,}")
print(f"   Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print("\n🔧 Changes made:")
print("   - Disabled mixed precision (using float32)")
print("   - Added BatchNormalization for stability")
print("   - Increased learning rates (0.01, 0.001)")
print("   - Reduced batch size to 32")

## 6. Phase 1: Train with Frozen Base

In [ ]:
# Compile model for Phase 1
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_PHASE1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks for Phase 1
callbacks_phase1 = [
    ModelCheckpoint(
        'efficientnetb0_phase1.keras',
        save_best_only=True,
        monitor='loss',
        verbose=1
    )
]

print("\n" + "="*80)
print("PHASE 1: Training EfficientNetB0 with frozen base model")
print("="*80)
print(f"Epochs: {EPOCHS_PHASE1}")
print(f"Learning rate: {LEARNING_RATE_PHASE1}")

start_time = time.time()

history_phase1 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
    verbose=1
)

phase1_time = (time.time() - start_time) / 60

print(f"\n✅ Phase 1 complete in {phase1_time:.2f} minutes")
print(f"   Average: {phase1_time/EPOCHS_PHASE1:.2f} minutes/epoch")
print(f"   Final training accuracy: {history_phase1.history['accuracy'][-1]:.4f}")

## 7. Phase 2: Fine-tune Last Layers

In [ ]:
# Unfreeze last N layers of base model
# EfficientNetB0 - unfreeze more layers for better adaptation
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30  # Unfreeze last 30 layers (increased from 20)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_PHASE2),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks for Phase 2
callbacks_phase2 = [
    ModelCheckpoint(
        KERAS_MODEL_PATH,
        save_best_only=True,
        monitor='loss',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='loss',
        patience=7,  # Increased from 5
        restore_best_weights=True,
        verbose=1
    )
]

print("\n" + "="*80)
print("PHASE 2: Fine-tuning EfficientNetB0 last layers")
print("="*80)
print(f"Epochs: {EPOCHS_PHASE2}")
print(f"Learning rate: {LEARNING_RATE_PHASE2}")
print(f"Unfrozen layers: {len([l for l in base_model.layers if l.trainable])}")

start_time = time.time()

history_phase2 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_phase2,
    verbose=1
)

phase2_time = (time.time() - start_time) / 60
total_training_time = phase1_time + phase2_time

print(f"\n✅ Phase 2 complete in {phase2_time:.2f} minutes")
print(f"   Average: {phase2_time/EPOCHS_PHASE2:.2f} minutes/epoch")
print(f"\n🎯 TOTAL TRAINING TIME: {total_training_time:.2f} minutes")

## 8. Visualize Training History

In [ ]:
# Combine histories
combined_history = {
    'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
    'accuracy': history_phase1.history['accuracy'] + history_phase2.history['accuracy']
}

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(combined_history['loss'], 'b-', linewidth=2, label='Training Loss')
axes[0].axvline(x=EPOCHS_PHASE1, color='r', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('EfficientNetB0 Training Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(combined_history['accuracy'], 'g-', linewidth=2, label='Training Accuracy')
axes[1].axvline(x=EPOCHS_PHASE1, color='r', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('EfficientNetB0 Training Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('efficientnetb0_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Final training accuracy: {combined_history['accuracy'][-1]:.4f}")
print(f"Final training loss: {combined_history['loss'][-1]:.4f}")

## 9. Evaluate on Test Set

In [ ]:
print("\n" + "="*80)
print("EVALUATING EFFICIENTNETB0 ON TEST SET")
print("="*80)

# Evaluate
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

# Get predictions
test_generator.reset()
y_pred = model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

# Calculate metrics
f1 = f1_score(y_true, y_pred_classes, average='weighted')

print(f"\n🎯 EFFICIENTNETB0 TEST SET RESULTS:")
print(f"   Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   F1-Score (weighted): {f1:.4f}")

# Classification report
print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
class_names = list(test_generator.class_indices.keys())
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - EfficientNetB0', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('efficientnetb0_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Convert to TFLite for Deployment

In [ ]:
# Convert to TFLite (standard)
print("Converting EfficientNetB0 to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

print(f"✅ TFLite model saved: {TFLITE_PATH}")
print(f"   Size: {os.path.getsize(TFLITE_PATH) / (1024*1024):.2f} MB")

# Convert to TFLite (Float16 - smaller size)
print("\nConverting to TFLite Float16...")
converter_float16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_float16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_float16.target_spec.supported_types = [tf.float16]
tflite_float16_model = converter_float16.convert()

with open(TFLITE_FLOAT16_PATH, 'wb') as f:
    f.write(tflite_float16_model)

print(f"✅ TFLite Float16 model saved: {TFLITE_FLOAT16_PATH}")
print(f"   Size: {os.path.getsize(TFLITE_FLOAT16_PATH) / (1024*1024):.2f} MB")
print(f"   Compression: {(1 - os.path.getsize(TFLITE_FLOAT16_PATH)/os.path.getsize(TFLITE_PATH))*100:.1f}% smaller")

## 11. Final Summary

In [ ]:
print("\n" + "="*80)
print("EFFICIENTNETB0 TRAINING COMPLETE")
print("="*80)

print(f"\n🎯 MODEL PERFORMANCE:")
print(f"  Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  F1-Score: {f1:.4f}")
print(f"  Test Loss: {test_loss:.4f}")

print(f"\n⏱️ TRAINING TIME:")
print(f"  Phase 1 (frozen): {phase1_time:.2f} minutes")
print(f"  Phase 2 (fine-tune): {phase2_time:.2f} minutes")
print(f"  Total: {total_training_time:.2f} minutes")

print(f"\n💾 SAVED FILES:")
print(f"  - {KERAS_MODEL_PATH}")
print(f"  - {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH)/(1024*1024):.2f} MB)")
print(f"  - {TFLITE_FLOAT16_PATH} ({os.path.getsize(TFLITE_FLOAT16_PATH)/(1024*1024):.2f} MB)")
print(f"  - efficientnetb0_training_history.png")
print(f"  - efficientnetb0_confusion_matrix.png")

print("\n" + "="*80)
print("✅ EFFICIENTNETB0 MODEL READY FOR ENSEMBLE!")
print("="*80)
print("\nNext steps:")
print("1. Copy", TFLITE_FLOAT16_PATH, "to the backend folder")
print("2. The backend will ensemble this with ResNet50 for improved accuracy")